In [5]:
import os
os.getcwd()

'c:\\Users\\Harsh Upadhyay\\OneDrive\\Desktop\\Nirikshan\\ml\\notebook'

In [8]:
from pathlib import Path

print(Path.cwd())

c:\Users\Harsh Upadhyay\OneDrive\Desktop\Nirikshan\ml\notebook


In [10]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

sys.path.insert(0, "../src/ml")

from visualize import *

In [13]:
import pandas as pd

df = pd.read_csv("../data/processed/Table6_Engineered.csv")

In [14]:
df.head()

,Ministry,Category,Project Name,Agency,Project Code,Legacy OCMS Code,PMGID,State,Original Cost (Rs. Crore),Revised Cost (Rs. Crore),...,is_multi_state,state_clean,ministry_freq_encoded,suspect_revised_cost,missing_approval_date,ministry_grouped,category_freq_encoded,agency_freq_encoded,state_freq_encoded,approval_year
0,Ministry of Civil Aviation,Aviation & Aviation Infrastructure,Construction of New Domestic Terminal Building...,Airport Authority of India [AAI],612786,N04000106,NaN,Andhra Pradesh,265.91,265.91,...,0,Andhra Pradesh,0.013125,0,0,Ministry of Civil Aviation,0.013125,0.012620,0.055023,2023.0
1,Ministry of Civil Aviation,Aviation & Aviation Infrastructure,Construction of New Integrated Terminal Buildi...,Airport Authority of India [AAI],701107,N04000091,4353.0,Andhra Pradesh,611.80,611.80,...,0,Andhra Pradesh,0.013125,0,0,Ministry of Civil Aviation,0.013125,0.012620,0.055023,2020.0
2,Ministry of Civil Aviation,Aviation & Aviation Infrastructure,Construction of New Domestic Terminal Building...,Airport Authority of India [AAI],701121,N04000103,NaN,Andhra Pradesh,347.15,347.15,...,0,Andhra Pradesh,0.013125,0,0,Ministry of Civil Aviation,0.013125,0.012620,0.055023,2022.0
3,Ministry of Civil Aviation,Aviation & Aviation Infrastructure,Guwahati Airport New Integrated Terminal Build...,Adani Airport Holdings Limited,706724,N04000083,9925.0,Assam,1712.00,2520.00,...,0,Assam,0.013125,0,0,Ministry of Civil Aviation,0.013125,0.000505,0.039879,2016.0
4,Ministry of Civil Aviation,Aviation & Aviation Infrastructure,Development of New Civil Enclave at Bihta,Airport Authority of India [AAI],612183,NaN,9918.0,Bihar,1413.00,1413.00,...,0,Bihar,0.013125,0,0,Ministry of Civil Aviation,0.013125,0.012620,0.051489,2024.0


In [19]:
from pathlib import Path
import sys
print("Working directory:", Path.cwd())
print("Python being used:", sys.executable)

Working directory: c:\Users\Harsh Upadhyay\OneDrive\Desktop\Nirikshan\ml\notebook
Python being used: c:\Users\Harsh Upadhyay\OneDrive\Desktop\Nirikshan\ml\.venv\Scripts\python.exe


In [22]:
from pathlib import Path
import pandas as pd
from ml.visualize import plot_cost_distribution, plot_physical_progress

PROJECT_ROOT = Path.cwd().parent  # agar CWD ml/notebook hai
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "Table6_Engineered.csv")

plot_cost_distribution(df)
plot_physical_progress(df)
print("Done — check ml/figures/ folder")

Done — check ml/figures/ folder


## Actual Plots and Visualizations
### Setup (data + models load)

In [23]:
from pathlib import Path
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split

from ml.visualize import *   # saare 10 plotting functions import ho jaenge

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "Table6_Engineered.csv"
MODELS_DIR = PROJECT_ROOT / "models"

df = pd.read_csv(DATA_PATH)

delay_clf = joblib.load(MODELS_DIR / "delay_classifier.joblib")
overrun_clf = joblib.load(MODELS_DIR / "cost_overrun_classifier.joblib")
slippage_reg = joblib.load(MODELS_DIR / "schedule_slippage_months_regressor.joblib")
overrun_reg = joblib.load(MODELS_DIR / "cost_overrun_pct_regressor.joblib")
kmeans = joblib.load(MODELS_DIR / "risk_clusters.joblib")
enc = joblib.load(MODELS_DIR / "encoders.joblib")

SAFE_FEATURES = [
    "ministry_freq_encoded","category_freq_encoded","agency_freq_encoded","state_freq_encoded",
    "is_multi_state","log_original_cost","planned_duration_months","approval_year",
    "project_age_months","Physical Progress (%)","log_cumulative_expenditure",
    "has_legacy_code","has_pmgid","missing_approval_date","original_is_outlier","cumulative_is_outlier",
]
X = df[SAFE_FEATURES]
print("Setup done —", df.shape[0], "rows loaded")

Setup done — 1981 rows loaded


EDA plots (5 files, sirf df chahiye)

In [24]:
plot_cost_distribution(df)              # 01_cost_distribution.png
plot_physical_progress(df)              # 02_physical_progress.png
plot_ministry_delay_rate(df)            # 03_ministry_delay_rate.png
plot_correlation_heatmap(df, SAFE_FEATURES)  # 04_correlation_heatmap.png
plot_outlier_boxplot(df)                # 05_cost_outliers_boxplot.png
print("5 EDA plots done")

5 EDA plots done


### Delay classifier plots (train.py jaisa hi split, taaki same test set mile):

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, df["is_delayed"], test_size=0.2, stratify=df["is_delayed"], random_state=42
)
proba = delay_clf.predict_proba(X_test)[:, 1]
preds = delay_clf.predict(X_test)

plot_confusion_matrix(y_test, preds, "Delay Classifier — Confusion Matrix", "06_delay_confusion_matrix")
plot_roc_curve(y_test, proba, "Delay Classifier — ROC Curve", "07_delay_roc_curve")
plot_feature_importance(delay_clf, SAFE_FEATURES, "Delay Classifier — Feature Importance", "08a_delay_feature_importance")
print("Delay classifier plots done")

Delay classifier plots done


### Cost overrun classifier plots:

In [26]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, df["is_cost_overrun"], test_size=0.2, stratify=df["is_cost_overrun"], random_state=42
)
proba2 = overrun_clf.predict_proba(X_test2)[:, 1]
preds2 = overrun_clf.predict(X_test2)

plot_confusion_matrix(y_test2, preds2, "Cost Overrun Classifier — Confusion Matrix", "06b_overrun_confusion_matrix")
plot_roc_curve(y_test2, proba2, "Cost Overrun Classifier — ROC Curve", "07b_overrun_roc_curve")
plot_feature_importance(overrun_clf, SAFE_FEATURES, "Cost Overrun Classifier — Feature Importance", "08b_overrun_feature_importance")
print("Cost overrun classifier plots done")

Cost overrun classifier plots done


### Regressor plots (sirf at-risk subset pe, jaisa hurdle-model design tha)

In [27]:
mask = df["is_delayed"] == 1
Xs, ys = X[mask], df["schedule_slippage_months"][mask]
_, Xte, _, yte = train_test_split(Xs, ys, test_size=0.2, random_state=42)
pred_slip = slippage_reg.predict(Xte)
plot_predicted_vs_actual(yte, pred_slip, "Slippage Regressor — Predicted vs Actual (months)", "09a_slippage_pred_vs_actual")

mask2 = df["is_cost_overrun"] == 1
Xs2, ys2 = X[mask2], df["cost_overrun_pct"][mask2]
_, Xte2, _, yte2 = train_test_split(Xs2, ys2, test_size=0.2, random_state=42)
pred_overrun = overrun_reg.predict(Xte2)
plot_predicted_vs_actual(yte2, pred_overrun, "Overrun % Regressor — Predicted vs Actual", "09b_overrun_pred_vs_actual")
print("Regressor plots done")

Regressor plots done


## Cluster plot

In [28]:
cluster_features = enc["cluster_features"]
X_cluster = df[cluster_features].fillna(df[cluster_features].median())
X_scaled = enc["cluster_scaler"].transform(X_cluster)

df["cluster"] = kmeans.predict(X_scaled)
df["risk_segment"] = df["cluster"].map(enc["risk_labels"])

plot_clusters_pca(X_scaled, df["cluster"], df["risk_segment"])
print("Cluster plot done — all 14 figures generated, check ml/figures/")

Cluster plot done — all 14 figures generated, check ml/figures/
